# 빅콘테스트 2026 – 제공 데이터 상세 탐색 (EDA)

신한카드 데이터 1·2와 SKT 유동인구 데이터를 뜯어보고 **의문점**을 눈으로 확인하는 노트북.

**사용 전 준비**
1. 구글 드라이브에 아래 파일을 올려둔다 (폴더는 자유, 아래 `설정` 셀에서 경로만 맞추면 됨)
   - `신한카드_빅콘테스트2026_데이터1.txt`, `신한카드_빅콘테스트2026_데이터2_수정.txt`
   - 유동인구 CSV 18개 (`flow_age_pop_2025MM.csv`, `flow_time_pop_…`, `flow_wkdy_pop_…`) → 한 폴더에
   - `weather_daily_2025H2.csv` (기상 일자료 통합본, 비 반응 확인용)
2. 런타임 → 모두 실행. 카드 데이터 읽는 데 1~2분 걸림.

| 파트 | 내용 |
|---|---|
| 0 | 설정·불러오기 |
| A | 신한카드 데이터 (A1~A10) |
| B | SKT 유동인구 데이터 (B1~B8) |
| C | 의문점 요약 |

## 0. 설정

In [ ]:
# 한글 폰트 설치 (코랩). 설치 후 재시작 없이 바로 등록한다.
import os, subprocess
if not os.path.exists('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'):
    subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== 경로 설정: 본인 드라이브 위치에 맞게 수정 =====
BASE     = '/content/drive/MyDrive/bigcontest2026'
CARD1    = f'{BASE}/신한카드_빅콘테스트2026_데이터1.txt'
CARD2    = f'{BASE}/신한카드_빅콘테스트2026_데이터2_수정.txt'
FLOW_DIR = f'{BASE}/flow'                       # 유동인구 CSV 18개가 있는 폴더
WEATHER  = f'{BASE}/weather_daily_2025H2.csv'

In [ ]:
import glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 120)

# 한글 폰트 등록
for p in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf']:
    if os.path.exists(p):
        fm.fontManager.addfont(p); plt.rc('font', family=fm.FontProperties(fname=p).get_name())
plt.rc('axes', unicode_minus=False)

# 그래프 공통 스타일 (얇은 막대, 옅은 격자)
BLUE, ORANGE, GRAY, INK, MUTED = '#2a78d6', '#eb6834', '#b5b3ad', '#0b0b0b', '#52514e'
REG_COLOR = {'서울 강남구': BLUE, '강원 춘천시': ORANGE, '강남': BLUE, '춘천': ORANGE}
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': GRAY,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': '#e6e5e0', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'lines.linewidth': 2, 'legend.frameon': False,
})
EOK = 1e8   # 억 원

In [ ]:
# ===== 카드 데이터 불러오기 (CP949, 탭 구분) =====
d1 = pd.read_csv(CARD1, sep='\t', encoding='cp949', dtype={'TA_YMD': str})
d2 = pd.read_csv(CARD2, sep='\t', encoding='cp949', dtype={'TA_YMD': str})
for d in (d1, d2):
    d['date'] = pd.to_datetime(d['TA_YMD'])
d1p = d1[d1['SEX_CCD'] != '법인']          # 데이터1 중 개인 결제만
print('데이터1', d1.shape, ' / 데이터2', d2.shape)
d1.head()

---
# A. 신한카드 데이터
## A1. 기본 구조 – 값의 종류

In [ ]:
for name, d in [('데이터1', d1), ('데이터2', d2)]:
    print(f'===== {name}')
    for c in d.columns:
        if c in ('TS_AT', 'USE_CNT', 'date'): continue
        vc = d[c].value_counts()
        show = vc.to_dict() if len(vc) <= 20 else f'(고유값 {len(vc)}개)'
        print(f'{c:12s} {len(vc):4d}개  {show}')
    print('결측:', int(d.isna().sum().sum()), ' / 기간:', d.TA_YMD.min(), '~', d.TA_YMD.max(), f'({d.TA_YMD.nunique()}일)')

**확인 포인트**
- `MCT_SGG_CD`(가맹점 지역)는 **'서울 강남구', '강원 춘천시' 두 값뿐** → 카드로는 동·상권 단위 분석 불가
- 정의서상 성별은 '남/여'인데 데이터1에는 **'법인'** 이 있음 (데이터2는 법인 제외)
- 업종: 데이터1 91개, 데이터2 90개 → 아래에서 차이 확인
- 고객 거주지(`CLN_SGG_CD`)는 시도 단위, **'전남광주' 통합 표기**, '정보없음' 존재

In [ ]:
print('데이터1에만 있는 업종:', set(d1.MCT_RY_CD) - set(d2.MCT_RY_CD))
print('데이터2에만 있는 업종:', set(d2.MCT_RY_CD) - set(d1.MCT_RY_CD))
pres = d1p.groupby(['MCT_RY_CD', 'MCT_SGG_CD']).TA_YMD.nunique().unstack()
print('\n한쪽 지역에만 있는 업종'); display(pres[pres.isna().any(axis=1)])

## A2. 데이터1 vs 데이터2 정합성 – "5건 미만 가림"은 어떻게 처리됐나?
5건 미만 칸을 **삭제**했다면, 데이터1(시간대로 쪼갬)과 데이터2(고객 거주지로 쪼갬)는 지워지는 칸이 달라서 합계가 어긋나야 한다.

In [ ]:
k = ['TA_YMD', 'MCT_SGG_CD', 'MCT_RY_CD', 'SEX_CCD', 'AGE_CCD']
g1 = d1p.groupby(k)[['TS_AT', 'USE_CNT']].sum()
g2 = d2.groupby(k)[['TS_AT', 'USE_CNT']].sum()
m = g1.join(g2, lsuffix='_1', rsuffix='_2', how='outer')

print('금액 합계  데이터1(개인):', f'{d1p.TS_AT.sum():,}', ' 데이터2:', f'{d2.TS_AT.sum():,}',
      ' 차이:', f'{d2.TS_AT.sum() - d1p.TS_AT.sum():,}원')
print('건수 합계  데이터1(개인):', f'{d1p.USE_CNT.sum():,}', ' 데이터2:', f'{d2.USE_CNT.sum():,}',
      f' 차이: {(d2.USE_CNT.sum()/d1p.USE_CNT.sum()-1)*100:.3f}%')
print('한쪽에만 있는 칸:', int(m.isna().any(axis=1).sum()))
m['건수차'] = m.USE_CNT_2 - m.USE_CNT_1
print('칸별 건수 차이 분포:', m['건수차'].value_counts().sort_index().to_dict())

In [ ]:
# 결제건수 값 분포 (5~40): 데이터2에는 아예 나오지 않는 값이 있다
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
rng = np.arange(5, 41)
for ax, (name, d) in zip(axes, [('데이터1', d1), ('데이터2', d2)]):
    vc = d.USE_CNT.value_counts().reindex(rng, fill_value=0)
    ax.bar(rng, vc.values, width=0.8, color=BLUE)
    zero = rng[vc.values == 0]
    ax.scatter(zero, np.zeros(len(zero)), marker='x', color=ORANGE, s=60, zorder=3, label='행이 하나도 없는 값')
    ax.set_title(f'{name} – 결제건수(USE_CNT) 값별 행 수'); ax.set_ylabel('행 수'); ax.legend(loc='upper right')
axes[-1].set_xlabel('결제건수'); axes[-1].set_xticks(rng)
plt.tight_layout(); plt.show()

**해석**
- 금액 합계는 두 데이터가 사실상 같음(30조 원 중 수천 원 차이) → 작은 칸을 **지운 게 아니라 다른 방식으로 처리**한 것으로 보임
- 결제건수는 7~10, 14~15, 20~21 같은 값이 데이터2에 **한 번도 안 나옴** → 건수가 가공(구간화·노이즈 등)됐을 가능성
- → **금액 중심으로 분석**, 건수·객단가(금액÷건수)는 주의. 처리 방식은 **주최 측 문의**

## A3. 업종 구성 – 골목상권과 무관한 업종이 얼마나 되나

In [ ]:
ind = d1p.groupby(['MCT_SGG_CD', 'MCT_RY_CD']).TS_AT.sum().unstack(0).fillna(0)
share = ind / ind.sum() * 100
NON_LOCAL = ['세금공과금', 'ZZ_나머지', '보험', '컴퓨터/소프트웨어', '쇼핑몰', '상품권/복권', '결제대행(PG)',
             '통신요금(이동,시내전화)', '통신요금(PC통신,무선호출)', '학교등록금']   # 비골목 후보 (팀 논의로 확정)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, r in zip(axes, ['서울 강남구', '강원 춘천시']):
    top = share[r].sort_values(ascending=False).head(20)[::-1]
    colors = [GRAY if i in NON_LOCAL else REG_COLOR[r] for i in top.index]
    ax.barh(top.index, top.values, color=colors, height=0.7)
    for y, v in enumerate(top.values): ax.text(v + 0.3, y, f'{v:.1f}%', va='center', fontsize=9, color=MUTED)
    ax.set_title(f'{r} – 업종별 개인 결제금액 비중 (상위 20)'); ax.set_xlabel('비중(%)')
fig.text(0.5, -0.01, '회색 = 골목상권과 무관한 업종 후보', ha='center', color=MUTED)
plt.tight_layout(); plt.show()

print('비골목 후보 업종 합계 비중(%):', share.loc[share.index.isin(NON_LOCAL)].sum().round(1).to_dict())

## A4. 'ZZ_나머지' – 정체불명 업종
춘천에서 특정 날짜에 하루 수십~백억 원대로 튄다.

In [ ]:
zz = d1p[d1p.MCT_RY_CD == 'ZZ_나머지'].groupby(['date', 'MCT_SGG_CD']).agg(amt=('TS_AT', 'sum'), cnt=('USE_CNT', 'sum'))
zz['객단가'] = zz.amt / zz.cnt
zc = zz.xs('강원 춘천시', level=1)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(zc.index, zc.amt / EOK, color=ORANGE, lw=1.5)
spk = zc.nlargest(10, 'amt').sort_index()   # 상위 10일만 표시
ax.scatter(spk.index, spk.amt / EOK, color=ORANGE, s=30, zorder=3)
for d, v in spk.amt.items(): ax.annotate(d.strftime('%m/%d'), (d, v / EOK), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=8, color=MUTED)
ax.set_title('춘천 ZZ_나머지 – 일별 결제금액'); ax.set_ylabel('억 원')
plt.tight_layout(); plt.show()

print('평소(중앙값) 하루:', round(zc.amt.median() / EOK, 1), '억 원 / 객단가', f'{zc.객단가.median():,.0f}원')
print('평소의 5배 넘는 날:', int((zc.amt > zc.amt.median() * 5).sum()), '일 (매달 중순·월말에 반복)')
display((spk.assign(억원=spk.amt / EOK, 객단가=spk.객단가.round(0))[['억원', 'cnt', '객단가']]).round(1))
print('\nZZ_나머지 시간대별 금액(억 원)')
display((d1p[d1p.MCT_RY_CD == 'ZZ_나머지'].groupby(['MCT_SGG_CD', 'TIME_GB']).TS_AT.sum().unstack() / EOK).round(0))

## A5. 일별 총매출 – 튀는 날은 무엇 때문인가

In [ ]:
daily_all = d1p.groupby(['date', 'MCT_SGG_CD']).TS_AT.sum().unstack() / EOK
daily_loc = d1p[~d1p.MCT_RY_CD.isin(NON_LOCAL)].groupby(['date', 'MCT_SGG_CD']).TS_AT.sum().unstack() / EOK

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, r in zip(axes, ['서울 강남구', '강원 춘천시']):
    ax.plot(daily_all.index, daily_all[r], color=GRAY, lw=1.2, label='전체 업종')
    ax.plot(daily_loc.index, daily_loc[r], color=REG_COLOR[r], lw=1.8, label='비골목 후보 제외')
    ax.set_title(f'{r} – 일별 개인 결제금액'); ax.set_ylabel('억 원'); ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

In [ ]:
# 튀는 날 상위 5일과 그날 가장 많이 늘어난 업종
def spike_drivers(region, n=5):
    s = daily_all[region]; base = s.rolling(15, center=True, min_periods=5).median()
    top = ((s - base) / base).nlargest(n)
    by = d1p[d1p.MCT_SGG_CD == region].groupby(['date', 'MCT_RY_CD']).TS_AT.sum().unstack(fill_value=0) / EOK
    rows = []
    for d, z in top.items():
        diff = by.loc[d] - by.rolling(15, center=True, min_periods=5).median().loc[d]
        rows.append({'날짜': d.date(), '평소 대비': f'{z*100:+.0f}%', '주도 업종': diff.idxmax(), '증가(억)': round(diff.max(), 0)})
    return pd.DataFrame(rows)
for r in ['서울 강남구', '강원 춘천시']:
    print('=====', r); display(spike_drivers(r))

## A6. 고객 거주지 – 강남 가맹점에서 결제한 비수도권 고객?
강남 커피전문점·할인점 결제의 상당 부분이 비수도권 거주 고객이다. 정말 강남에 와서 쓴 걸까?

In [ ]:
METRO = ['서울', '경기', '인천']
g = d2[d2.MCT_SGG_CD == '서울 강남구'].groupby(['MCT_RY_CD', 'CLN_SGG_CD']).TS_AT.sum().unstack(fill_value=0)
nonmetro = (g.drop(columns=[c for c in METRO + ['정보없음'] if c in g]).sum(axis=1) / g.sum(axis=1) * 100)
big = g.sum(axis=1).sort_values(ascending=False).head(25).index
x = nonmetro[big].sort_values()

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(x.index, x.values, color=[GRAY if i in NON_LOCAL else BLUE for i in x.index], height=0.7)
for y, v in enumerate(x.values): ax.text(v + 0.5, y, f'{v:.0f}%', va='center', fontsize=9, color=MUTED)
ax.set_title('강남 가맹점 – 비수도권 거주 고객 결제 비중 (금액 상위 25 업종)'); ax.set_xlabel('비수도권 고객 비중(%)')
plt.tight_layout(); plt.show()

## A7. 비 오는 날 반응 – 수도권 고객 vs 비수도권 고객 (강남)
비수도권 고객 결제가 **강남 비에 반응하지 않는다면**, 실제로는 강남 밖에서 일어난 결제(본사 일괄 등록 가맹점 등)일 가능성이 크다.

방법: 7/22 이후(쿠폰 지급 첫 주 제외), 추석 연휴(10/3~12) 제외. 같은 달·같은 요일 평균과의 차이(로그)를 구한 뒤 20mm 이상 비 온 날과 마른 날(1mm 미만)을 비교.

In [ ]:
w = pd.read_csv(WEATHER, parse_dates=['date'])
rain_gn = w[w.station.str.contains('강남')].set_index('date').rain_mm
rain_cc = w[w.station.str.contains('춘천\\(101')].set_index('date').rain_mm

def rain_effect(series, rain, heavy=20):
    s = series[(series.index >= '2025-07-22') & ~series.index.isin(pd.date_range('2025-10-03', '2025-10-12'))]
    r = rain.reindex(s.index).fillna(0)
    lg = np.log(s); resid = lg - lg.groupby([s.index.month, s.index.dayofweek]).transform('mean')
    return (np.exp(resid[r >= heavy].mean() - resid[r < 1].mean()) - 1) * 100, int((r >= heavy).sum())

x = d2[d2.MCT_SGG_CD == '서울 강남구'].assign(grp=lambda t: np.where(t.CLN_SGG_CD.isin(METRO), '수도권 고객', '비수도권 고객'))
rows = []
for ind_ in ['커피전문점', '할인점/슈퍼마켓/양판점', '한식', '편의점', '의복/의류', '일식']:
    gg = x[x.MCT_RY_CD == ind_].groupby(['date', 'grp']).TS_AT.sum().unstack()
    for grp in ['수도권 고객', '비수도권 고객']:
        eff, n = rain_effect(gg[grp], rain_gn); rows.append({'업종': ind_, '고객': grp, '효과(%)': eff, '비 온 날': n})
res = pd.DataFrame(rows); piv = res.pivot(index='업종', columns='고객', values='효과(%)')[['수도권 고객', '비수도권 고객']]

fig, ax = plt.subplots(figsize=(11, 4.5)); yy = np.arange(len(piv)); h = 0.36
ax.barh(yy + h/2, piv['수도권 고객'], height=h - 0.04, color=BLUE, label='수도권 고객')
ax.barh(yy - h/2, piv['비수도권 고객'], height=h - 0.04, color=ORANGE, label='비수도권 고객')
ax.axvline(0, color=MUTED, lw=1); ax.set_yticks(yy, piv.index)
ax.set_title(f"강남 – 20mm 이상 비 온 날 매출 변화 (비 온 날 {res['비 온 날'].iloc[0]}일, 참고용)"); ax.set_xlabel('마른 날 대비 변화(%)'); ax.legend()
plt.tight_layout(); plt.show()
display(piv.round(1))

**해석**: 수도권 고객은 비 오면 줄고, 비수도권 고객은 거의 그대로 → 비수도권 결제 상당수가 강남 현장 결제가 아닐 가능성
→ 강남 민감도가 **희석돼 과소추정**될 수 있음. 강남은 수도권 고객만 따로 보는 방안 검토. (단, 비 온 날 9일뿐 → 참고용)

아래 표: 수도권 고객만 봐도 강남 커피는 −2%대 → −3%대 정도로만 커짐. **춘천(−11%)이 훨씬 민감하다는 결론은 유지**됨.

In [ ]:
# 참고: 같은 방식으로 본 1차 비교 – 전체 고객 vs 수도권 고객만 (강남), 춘천 전체
rows = []
for ind_ in ['커피전문점', '일식', '의복/의류', '한식', '편의점', '할인점/슈퍼마켓/양판점']:
    gn_all = d2[(d2.MCT_SGG_CD == '서울 강남구') & (d2.MCT_RY_CD == ind_)].groupby('date').TS_AT.sum()
    gn_met = x[(x.MCT_RY_CD == ind_) & (x.grp == '수도권 고객')].groupby('date').TS_AT.sum()
    cc_all = d2[(d2.MCT_SGG_CD == '강원 춘천시') & (d2.MCT_RY_CD == ind_)].groupby('date').TS_AT.sum()
    rows.append({'업종': ind_, '강남 전체': rain_effect(gn_all, rain_gn)[0], '강남 수도권 고객만': rain_effect(gn_met, rain_gn)[0],
                 '춘천 전체': rain_effect(cc_all, rain_cc)[0]})
display(pd.DataFrame(rows).set_index('업종').round(1))

## A8. 법인 결제 비중 – 빼도 되나?

In [ ]:
corp = d1.assign(is_corp=d1.SEX_CCD == '법인').groupby(['MCT_SGG_CD', 'MCT_RY_CD', 'is_corp']).TS_AT.sum().unstack(fill_value=0)
corp['법인%'] = corp[True] / corp.sum(axis=1) * 100
food = ['한식', '중식', '일식', '양식', '기타요식', '커피전문점', '유흥업소', '주류판매', '제과점', '패스트푸드', '호텔/콘도', '주유소']
p = corp['법인%'].unstack(0).loc[food].sort_values('서울 강남구')

fig, ax = plt.subplots(figsize=(10, 5.5)); yy = np.arange(len(p)); h = 0.36
ax.barh(yy + h/2, p['서울 강남구'], height=h - 0.04, color=BLUE, label='강남')
ax.barh(yy - h/2, p['강원 춘천시'], height=h - 0.04, color=ORANGE, label='춘천')
ax.set_yticks(yy, p.index); ax.set_xlabel('법인카드 비중(%)'); ax.set_title('업종별 법인카드 결제 비중 (데이터1)'); ax.legend()
plt.tight_layout(); plt.show()

**해석**: 강남 음식점 매출의 40% 이상이 법인카드 → 회식·점심도 소상공인 실제 매출. '법인 제외' 결정 재검토 필요

## A9. 요일·시간대 패턴

In [ ]:
dw = daily_loc.groupby(daily_loc.index.dayofweek).mean(); dw = dw / dw.mean()
tg = d1p[~d1p.MCT_RY_CD.isin(NON_LOCAL)].groupby(['MCT_SGG_CD', 'TIME_GB']).TS_AT.sum().unstack(0); tg = tg / tg.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4)); h = 0.36
for ax, df, lab, title in [(axes[0], dw, ['월', '화', '수', '목', '금', '토', '일'], '요일별 매출 (평균=1, 비골목 후보 제외)'),
                           (axes[1], tg, list(tg.index), '시간대별 매출 비중(%)')]:
    xx = np.arange(len(df))
    ax.bar(xx - h/2, df['서울 강남구'], width=h - 0.04, color=BLUE, label='강남')
    ax.bar(xx + h/2, df['강원 춘천시'], width=h - 0.04, color=ORANGE, label='춘천')
    ax.set_xticks(xx, lab); ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

**해석**: 비골목 후보 업종을 빼면 강남도 주말 매출이 평일과 비슷하거나 높음 → 원자료의 '강남 일요일 급감'은 세금공과금 등 사무성 결제 때문. 업종 정리 전후로 패턴이 크게 달라지므로 **제외 업종을 먼저 확정**해야 함

## A10. 업종별 데이터가 있는 날 수 – 희소한 업종

In [ ]:
days = d1p.groupby(['MCT_RY_CD', 'MCT_SGG_CD']).TA_YMD.nunique().unstack()
sparse = days[(days < 184).any(axis=1)].sort_values('강원 춘천시')
display(sparse)
print('184일 중 결제가 있는 날이 적은 업종은 일 단위 분석에 부적합 → 묶거나 제외')

---
# B. SKT 유동인구 데이터
## B1. 불러오기 – 표 3종 × 월

In [ ]:
def load_flow(kind):
    fs = sorted(glob.glob(f'{FLOW_DIR}/flow_{kind}_pop_*.csv'))
    print(kind, [os.path.basename(f)[-10:-4] for f in fs])
    df = pd.concat([pd.read_csv(f, sep='|', encoding='utf-8-sig', dtype={'BLOCK_CD': str, 'STD_YM': str}) for f in fs])
    df['sgg'] = df.BLOCK_CD.str[:5]
    df['region'] = df.sgg.map({'11230': '강남', '32010': '춘천'}).fillna('기타')
    return df
FA, FT, FW = load_flow('age'), load_flow('time'), load_flow('wkdy')
AV = [c for c in FA.columns if 'FLOW_POP' in c]; TV = [c for c in FT.columns if c.startswith('TMST')]; WV = [c for c in FW.columns if c.startswith('FLOW_POP')]
FA.head()

## B2. 소지역코드(BLOCK_CD)와 경계 밖 격자
앞 5자리가 통계청 시군구 코드(강남 11230, 춘천 32010). 그 외 코드는 경계에 걸친 다른 시군구 격자.

In [ ]:
print('BLOCK_CD 길이:', FA.BLOCK_CD.str.len().unique())
print('지역별 고유 BLOCK_CD 수:', FA.groupby('region').BLOCK_CD.nunique().to_dict())
print('\n강남·춘천 외 코드 (월별 격자 수)')
display(FA[FA.region == '기타'].groupby(['STD_YM', 'sgg']).size().unstack(fill_value=0))
print('→ 강남 2,030개 / 춘천 746개 소지역 = 통계청 집계구 코드일 가능성 (주최 측 확인)')

## B3. 표마다 격자 수가 다르다

In [ ]:
cnt = pd.concat({'성연령': FA.groupby(['STD_YM', 'region']).size(), '시간대': FT.groupby(['STD_YM', 'region']).size(),
                 '요일': FW.groupby(['STD_YM', 'region']).size()}, axis=1).unstack()
display(cnt.xs('춘천', axis=1, level=1).add_suffix('_춘천').join(cnt.xs('강남', axis=1, level=1).add_suffix('_강남')))

m = '202509'; key = lambda d: set(zip(d[d.STD_YM == m].X_COORD.round(1), d[d.STD_YM == m].Y_COORD.round(1)))
ka, kt, kw = key(FA), key(FT), key(FW)
print(f'{m} 격자 수  성연령 {len(ka):,} / 시간대 {len(kt):,} / 요일 {len(kw):,} / 세 표 공통 {len(ka & kt & kw):,}')

## B4. 숫자의 의미 – 사람 수가 아니다
시간대 표의 24시간을 더하면 하루 총량과 같다 → 시간 값은 '그 시각 체류 인구'가 아니라 **하루 유동을 시간별로 나눈 값**.

In [ ]:
FA2, FT2, FW2 = (d[d.region != '기타'] for d in (FA, FT, FW))
tot = pd.DataFrame({'성연령 합': FA2.groupby(['STD_YM', 'region'])[AV].sum().sum(axis=1),
                    '시간대 24시간 합': FT2.groupby(['STD_YM', 'region'])[TV].sum().sum(axis=1),
                    '요일 평균': FW2.groupby(['STD_YM', 'region'])[WV].sum().sum(axis=1) / 7})
display((tot / 1e4).round(0).rename(columns=lambda c: c + '(만)'))
print('강남 격자 합 ≈ 하루 700만대 → 한 사람이 여러 격자에 중복 집계됨. 절대값이 아니라 상대 비교로만 사용')

## B5. 값의 크기 – 작은 격자가 대부분 (특히 춘천)

In [ ]:
cell = FA2[FA2.STD_YM == '202509'].assign(tot=lambda d: d[AV].sum(axis=1))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bins = np.logspace(-2, 4.2, 60)
for r in ['강남', '춘천']:
    axes[0].hist(cell[cell.region == r].tot, bins=bins, color=REG_COLOR[r], alpha=0.55, label=r)
axes[0].set_xscale('log'); axes[0].axvline(1, color=MUTED, lw=1, ls='--'); axes[0].set_title('격자별 하루 유동인구 분포 (2025-09, 로그축)')
axes[0].set_xlabel('격자 하루 유동인구'); axes[0].set_ylabel('격자 수'); axes[0].legend()

v = FA2[AV].values.ravel(); v = v[(v > 0) & (v < 0.6)]
u, c = np.unique(np.round(v, 3), return_counts=True)
axes[1].bar(u, c, width=0.012, color=BLUE); axes[1].set_title('0~0.6 구간 값 – 0.025 단위로 끊김'); axes[1].set_xlabel('값'); axes[1].set_ylabel('개수')
plt.tight_layout(); plt.show()

for r in ['강남', '춘천']:
    s = cell[cell.region == r].tot.sort_values(ascending=False)
    print(f'{r}: 격자 {len(s):,}개 / 하루 1 미만 {(s < 1).mean()*100:.0f}% / 상위 1% 격자가 전체의 {s.head(len(s)//100).sum()/s.sum()*100:.0f}%')

## B6. 지도로 보기 – 격자 위치와 크기

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, r in zip(axes, ['강남', '춘천']):
    c = cell[cell.region == r]
    sc = ax.scatter(c.X_COORD, c.Y_COORD, c=np.log10(c.tot + 0.01), s=2 if r == '강남' else 0.6, cmap='Blues', vmin=-1, vmax=3.5)
    ax.set_aspect('equal'); ax.set_title(f'{r} – 격자별 하루 유동인구 (log10, 2025-09)'); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    other = FA[(FA.STD_YM == '202509') & (FA.region == '기타')]
    near = other[(other.X_COORD.between(c.X_COORD.min() - 2000, c.X_COORD.max() + 2000)) & (other.Y_COORD.between(c.Y_COORD.min() - 2000, c.Y_COORD.max() + 2000))]
    ax.scatter(near.X_COORD, near.Y_COORD, s=12, color=ORANGE, label='경계 밖 코드 격자'); ax.legend(loc='lower right')
plt.colorbar(sc, ax=axes, shrink=0.6, label='log10(유동인구)'); plt.show()

## B7. 시간대·요일·연령 패턴

In [ ]:
hr = FT2.groupby('region')[TV].sum(); hr = hr.div(hr.sum(axis=1), axis=0) * 100
wk = FW2.groupby('region')[WV].sum(); wk = wk.div(wk.mean(axis=1), axis=0)
ag = FA2.groupby('region')[AV].sum()
age = pd.DataFrame({g: ag[[c for c in AV if c.endswith(g)]].sum(axis=1) for g in ['10G', '20G', '30G', '40G', '50G', '60GU']})
age = age.div(age.sum(axis=1), axis=0) * 100; age.columns = ['10대', '20대', '30대', '40대', '50대', '60대+']

fig, axes = plt.subplots(1, 3, figsize=(17, 4)); h = 0.36
for r in ['강남', '춘천']: axes[0].plot(range(24), hr.loc[r].values, color=REG_COLOR[r], label=r)
axes[0].set_xticks(range(0, 24, 3)); axes[0].set_title('시간대별 비중(%)'); axes[0].set_xlabel('시'); axes[0].legend()
for ax, df, lab, title in [(axes[1], wk, ['월', '화', '수', '목', '금', '토', '일'], '요일별 (평균=1)'), (axes[2], age, list(age.columns), '연령 비중(%)')]:
    xx = np.arange(df.shape[1])
    ax.bar(xx - h/2, df.loc['강남'], width=h - 0.04, color=BLUE, label='강남'); ax.bar(xx + h/2, df.loc['춘천'], width=h - 0.04, color=ORANGE, label='춘천')
    ax.set_xticks(xx, lab); ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()
print('주의: 유동인구 연령은 60대 이상이 하나로 묶임 → 카드(60·70·80·90대)와 비교 시 60대+로 맞출 것')

## B8. 주말이 0인 격자 – 업무지구인가, 결측인가

In [ ]:
z = FW2[(FW2.FLOW_POP_CNT_SAT == 0) & (FW2.FLOW_POP_CNT_SUN == 0) & (FW2[WV[:5]].mean(axis=1) > 1)]
print('평일 평균 1 초과인데 토·일 모두 0인 격자-월:', f'{len(z):,}', '/ 지역별', z.region.value_counts().to_dict())
display(z.sort_values('FLOW_POP_CNT_WED', ascending=False).head(10)[['STD_YM', 'region', 'BLOCK_CD'] + WV])

---
# C. 의문점 요약

| # | 데이터 | 의문점 | 근거 셀 | 할 일 |
|---|---|---|---|---|
| 1 | 카드 | 지역이 시군구 2개뿐 → 상권 단위 측정 불가 | A1 | 두 층 구조 확정 |
| 2 | 카드 | 5건 미만 처리 방식? 금액 합계는 일치, 건수는 가공 흔적 | A2 | 금액 중심 분석, **주최 측 문의** |
| 3 | 카드 | 'ZZ_나머지'(강남 21%, 춘천 25%)의 정체, 춘천 15일·월말 급증 | A3, A4 | 제외 + **주최 측 문의** |
| 4 | 카드 | 강남 매출 절반이 세금공과금·ZZ_나머지 등 비골목 업종 | A3, A5 | 업종 매핑 표에서 제외 확정 |
| 5 | 카드 | 강남 비수도권 고객 결제가 비에 반응 안 함 → 본사 등록 가맹점 혼입? | A6, A7 | 강남은 수도권 고객만 분석 검토, **주최 측 문의** |
| 6 | 카드 | 음식점 법인카드 비중 40%+ → 법인 제외가 맞나? | A8 | 팀 논의 |
| 7 | 카드 | 결제일이 적은 희소 업종 | A10 | 묶거나 제외 |
| 8 | 유동 | 표 3종 격자 목록 불일치 | B3 | 결합 기준 정하기 |
| 9 | 유동 | 값은 사람 수가 아님(중복 집계), 시간 값 = 하루 총량의 시간 배분 | B4 | 상대 비교로만 사용 |
| 10 | 유동 | 작은 격자 다수·0.025 단위 끊김 (춘천 47%가 1 미만) | B5 | 50m 대신 묶어서 사용 |
| 11 | 유동 | BLOCK_CD = 집계구 코드? | B2 | **주최 측 문의** → SGIS 연결 |
| 12 | 유동 | 경계 밖 시군구 격자 혼입 | B2, B6 | 제외 |
| 13 | 유동 | 주말 0 격자 | B8 | 업무지구 vs 결측 확인 |
| 14 | 유동 | 연령 60대+ 통합 | B7 | 카드와 구간 맞추기 |